# 3.46 — Feature Selection

Feature selection chooses which columns a learner is allowed to use, then judges that choice by validation performance plus the cost of carrying features. In this lesson, we build filter, wrapper, and embedded selection from scratch with NumPy, keeping the central contract visible: a smaller training loss is not enough unless the subset also survives validation, cost, and stability checks.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build feature selection one idea at a time. Run each cell in order and read the printed intermediate values — every score is decomposed into raw fit, feature cost, and final decision quantity. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, masks, least-squares, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic examples.

### 1. The selection objective: validation risk plus feature cost

Feature selection asks for a subset $S$ of columns, not just a fitted model. The lesson's core score is

$$S^*=\arg\min_{S\subseteq\{1,\ldots,d\}} R_{val}(f_S)+\lambda|S|.$$

The first term measures future-facing validation loss for the model trained on subset $S$; the second charges for using more columns. This prevents a convenient but noisy feature from winning merely because it lowers one training number.

In [ ]:
losses_w = np.array([0.191, 0.070, 0.471])  # three verified per-example validation losses.
risk_w = float(losses_w.mean())  # empirical validation risk R_S.
print("losses:", losses_w)  # inspect the pieces before averaging.
print("R_S:", round(risk_w, 3))  # (0.191 + 0.070 + 0.471) / 3 = 0.244.
assert round(risk_w, 3) == 0.244  # lesson number from the source block.

▶ What you'll see: three tiny losses average to 0.244, the raw validation term.

In [ ]:
cost_w = 0.090  # complexity, regularization, or operational cost for this subset.
score_w = risk_w + cost_w  # selection score = validation risk plus cost.
print("risk:", round(risk_w, 3), "cost:", round(cost_w, 3), "score:", round(score_w, 3))
assert round(score_w, 3) == 0.334  # 0.244 + 0.090.

▶ What you'll see: the decision score is 0.334, not the prettier raw 0.244.

In [ ]:
plt.figure(figsize=(4.4, 3))  # compact score decomposition.
plt.bar(["R_val", "cost", "total"], [risk_w, cost_w, score_w], color=["teal", "orange", "purple"])
plt.title("1: feature-subset score = fit + cost")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the total bar is visibly larger than the raw validation-risk bar because selected features are not free.

*Why it's done this way:* Validation risk estimates how the subset behaves on data not used to fit parameters, while $\lambda|S|$ expresses the mathematical preference for simpler, cheaper, less variable models. Minimizing their sum forces the learner to justify each extra feature by enough validation gain to pay its cost.

### 2. Filter selection: score features before fitting a full model

A filter method ranks columns by a statistic computed directly from data, such as absolute correlation with the target. It is fast because it does not refit a model for every subset. The tradeoff is that a univariate score can miss interactions: a feature may look weak alone but matter with another feature.

In [ ]:
X_w = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.]])
y_w = np.array([0.2, 1.0, 2.1, 2.9, 4.2, 5.1])
print("X shape:", X_w.shape, "y mean:", round(float(y_w.mean()), 3))

▶ What you'll see: six examples with three candidate features and a roughly increasing target.

In [ ]:
Xc_w = X_w - X_w.mean(axis=0)  # center each feature so dot products measure co-movement.
yc_w = y_w - y_w.mean()  # center the target for the same reason.
num_w = Xc_w.T @ yc_w  # covariance-like numerators.
den_w = np.sqrt(np.sum(Xc_w ** 2, axis=0) * np.sum(yc_w ** 2))  # length products.
corr_w = num_w / den_w  # Pearson correlations from scratch.
print("correlations:", np.round(corr_w, 3))
assert np.argmax(np.abs(corr_w)) == 0  # feature 0 is the strongest filter feature.

▶ What you'll see: feature 0 has the largest absolute correlation with the target.

In [ ]:
order_w = np.argsort(np.abs(corr_w))[::-1]  # rank features by absolute filter score.
plt.figure(figsize=(4.6, 3))
plt.bar([f"x{j}" for j in range(X_w.shape[1])], np.abs(corr_w), color="seagreen")
plt.title("2: filter scores by |correlation|")
plt.ylabel("absolute correlation")
plt.show()
print("filter order:", order_w)

▶ What you'll see: the tallest bar is selected first by the filter rule.

*Why it's done this way:* Correlation divides feature-target co-movement by both vector lengths, so the score is about direction rather than raw units. Because this ignores the downstream model, it is a cheap screening device: useful when $d$ is large, but not a guarantee that the best individual feature creates the best subset.

### 3. Wrapper selection: evaluate subsets by validation score

A wrapper method treats feature choice as a search problem around a model. Here we fit tiny linear regressions by least squares and score every subset using validation mean squared error plus the same $\lambda|S|$ penalty. This is slower than a filter, but it directly evaluates the model we plan to use.

In [ ]:
Xtr_w = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.]])
ytr_w = np.array([0.1, 1.1, 2.0, 2.9, 4.0, 5.2])
Xval_w = np.array([[1.5, 0.5, 0.0], [3.5, 1.5, 1.0], [4.5, 2.0, 0.0]])
yval_w = np.array([1.55, 3.55, 4.75])
subsets_w = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2), (0, 1, 2)]
print("candidate subsets:", subsets_w)

▶ What you'll see: all nonempty subsets of three features are candidates.

In [ ]:
scores_wrap_w = []
for S_w in subsets_w:
    A_w = np.c_[np.ones(len(Xtr_w)), Xtr_w[:, S_w]]  # add intercept to selected columns.
    beta_w = np.linalg.lstsq(A_w, ytr_w, rcond=None)[0]  # least-squares fit from scratch.
    Aval_w = np.c_[np.ones(len(Xval_w)), Xval_w[:, S_w]]  # same selected columns on validation rows.
    mse_w = float(np.mean((yval_w - Aval_w @ beta_w) ** 2))  # validation risk.
    total_w = mse_w + 0.03 * len(S_w)  # penalized subset score.
    scores_wrap_w.append(total_w)
    print(S_w, "val mse=", round(mse_w, 3), "score=", round(total_w, 3))
best_idx_w = int(np.argmin(scores_wrap_w))
print("best wrapper subset:", subsets_w[best_idx_w])

▶ What you'll see: each subset gets its own validation-based decision score.

In [ ]:
plt.figure(figsize=(6, 3))
plt.bar([str(s) for s in subsets_w], scores_wrap_w, color="slateblue")
plt.xticks(rotation=35)
plt.ylabel("validation MSE + 0.03|S|")
plt.title("3: wrapper subset search")
plt.show()
assert subsets_w[best_idx_w] == (0,)  # on this toy split, cost makes the one-feature subset win.

▶ What you'll see: the best bar balances fit and subset size rather than taking every feature automatically.

*Why it's done this way:* The wrapper score uses the actual validation loss of $f_S$, so interactions between selected columns can matter. The penalty then asks whether an added column improves validation enough to overcome its cost; otherwise, lower-dimensional subsets win.

### 4. Embedded selection: let the fitting objective shrink features

Embedded methods perform selection while fitting the model. A simple from-scratch version is soft-thresholding: fit least-squares coefficients, then shrink small coefficients toward exactly zero. This mirrors the L1 idea behind lasso-style selection, where the objective rewards predictive fit but charges each coefficient for being nonzero.

In [ ]:
A_emb_w = np.c_[np.ones(len(Xtr_w)), Xtr_w]  # intercept plus all features.
beta_ls_w = np.linalg.lstsq(A_emb_w, ytr_w, rcond=None)[0]  # ordinary least squares coefficients.
print("least-squares coefficients:", np.round(beta_ls_w, 3))

▶ What you'll see: feature coefficients have different magnitudes; small ones are easiest to remove.

In [ ]:
threshold_w = 0.07  # L1-like shrinkage threshold applied only to feature coefficients.
coef_w = beta_ls_w[1:]  # exclude intercept from selection.
shrunk_w = np.sign(coef_w) * np.maximum(np.abs(coef_w) - threshold_w, 0.0)  # soft threshold.
selected_emb_w = np.where(np.abs(shrunk_w) > 0)[0]  # nonzero coefficients survive.
print("shrunk coefficients:", np.round(shrunk_w, 3))
print("embedded selected features:", selected_emb_w)
assert np.array_equal(selected_emb_w, np.array([0, 1]))

▶ What you'll see: the noisy third feature is shrunk to zero while the stronger coefficients remain.

In [ ]:
plt.figure(figsize=(5, 3))
xpos_w = np.arange(3)
plt.bar(xpos_w - 0.18, coef_w, width=0.36, label="least squares", color="gray")
plt.bar(xpos_w + 0.18, shrunk_w, width=0.36, label="shrunk", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(xpos_w, ["x0", "x1", "x2"])
plt.title("4: embedded shrinkage selects features")
plt.legend()
plt.show()

▶ What you'll see: the shrinkage bars are smaller, and one drops exactly to zero.

*Why it's done this way:* Penalizing coefficient magnitude changes the optimization geometry so weak coordinates are not merely made small; they can become zero. That zero is the embedded feature-selection decision, produced as part of fitting rather than by a separate ranking or subset search.

### 5. Gaps and stability: do not over-read tiny wins

After scoring subsets, the last question is whether the winning gap is meaningful. The source lesson compares a baseline score 0.334 with a more flexible alternative 0.374, then considers a stabilized version at 80% of the baseline score. The arithmetic is simple, but the modeling lesson is deep: a small validation gap can vanish under resampling noise, while stable improvements are safer to carry forward.

In [ ]:
baseline_w = 0.334  # validated subset score.
flexible_w = 0.374  # more flexible alternative's decision score.
gap_w = flexible_w - baseline_w  # absolute evidence for preferring baseline.
rel_gap_w = gap_w / flexible_w  # scale-aware evidence.
print("gap:", round(gap_w, 3), "relative gap:", round(rel_gap_w, 3))
assert round(gap_w, 3) == 0.040
assert round(rel_gap_w, 3) == 0.107

▶ What you'll see: the baseline beats the flexible alternative by 0.040, about 10.7% of the alternative score.

In [ ]:
stable_w = 0.80 * baseline_w  # stabilizing knob reduces the decision score by 20%.
choices_w = np.array([baseline_w, flexible_w, stable_w])
labels_w = np.array(["baseline", "flexible", "stabilized"])
winner_w = labels_w[int(np.argmin(choices_w))]
print("scores:", np.round(choices_w, 3))
print("winner:", winner_w)
assert round(stable_w, 3) == 0.267
assert winner_w == "stabilized"

▶ What you'll see: the stabilized score is the lowest of the three toy decisions.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(labels_w, choices_w, color=["gray", "orange", "seagreen"])
plt.ylabel("decision score (lower is better)")
plt.title("5: final feature-selection comparison")
plt.show()

▶ What you'll see: the stabilized bar sits below the baseline and flexible alternatives.

*Why it's done this way:* Feature selection is a decision under uncertainty, not just a leaderboard. The absolute gap measures evidence, the relative gap checks its scale, and stability asks whether the selected subset is likely to remain useful when the sample changes.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small arrays, prints intermediate values with inline `# ->` checks, draws one picture, and includes an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Selection score is validation risk plus feature cost

Feature selection ranks subsets by a decision score. First average validation losses, then add the cost of carrying selected columns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_losses = np.array([0.16, 0.22, 0.18, 0.24, 0.20, 0.30])
print("validation losses:", t1_losses.tolist())  # -> [0.16, 0.22, 0.18, 0.24, 0.2, 0.3]
t1_risk = float(t1_losses.mean())
print("validation risk:", round(t1_risk, 3))  # -> 0.217
t1_subset_size = 3
print("selected feature count:", t1_subset_size)  # -> 3
t1_lambda = 0.04
print("cost per feature:", t1_lambda)  # -> 0.04
t1_cost = t1_lambda * t1_subset_size
print("feature cost:", round(t1_cost, 3))  # -> 0.12
t1_score = t1_risk + t1_cost
print("selection score:", round(t1_score, 3))  # -> 0.337

plt.figure(figsize=(4.4, 3))
plt.bar(["risk", "cost", "total"], [t1_risk, t1_cost, t1_score], color=["teal", "orange", "purple"])
plt.ylabel("score")
plt.title("Toy 1 · fit plus feature cost")
plt.show()
assert round(t1_score, 3) == 0.337

▶ What you'll see: the subset's raw validation risk `0.217` becomes a larger decision score `0.337` after cost.

### ✍️ Toy 2 · A filter ranks columns by absolute correlation

A filter score looks at each column before fitting the final model. Here the strongest column is the one whose centered values move most with the centered target.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_X = np.array([[0., 1., 1.], [1., 1., 0.], [2., 2., 1.], [3., 2., 0.], [4., 3., 1.], [5., 3., 0.]])
print("feature matrix shape:", t2_X.shape)  # -> (6, 3)
t2_y = np.array([0.1, 1.2, 2.1, 3.0, 4.2, 5.1])
print("target:", t2_y.tolist())  # -> [0.1, 1.2, 2.1, 3.0, 4.2, 5.1]
t2_X_mean = t2_X.mean(axis=0)
print("feature means:", np.round(t2_X_mean, 3).tolist())  # -> [2.5, 2.0, 0.5]
t2_y_mean = float(t2_y.mean())
print("target mean:", round(t2_y_mean, 3))  # -> 2.617
t2_X_centered = t2_X - t2_X_mean
print("centered first row:", t2_X_centered[0].tolist())  # -> [-2.5, -1.0, 0.5]
t2_y_centered = t2_y - t2_y_mean
print("centered target:", np.round(t2_y_centered, 3).tolist())  # -> [-2.517, -1.417, -0.517, 0.383, 1.583, 2.483]
t2_num = t2_X_centered.T @ t2_y_centered
print("correlation numerators:", np.round(t2_num, 3).tolist())  # -> [17.45, 8.0, -1.45]
t2_den = np.sqrt(np.sum(t2_X_centered ** 2, axis=0) * np.sum(t2_y_centered ** 2))
print("correlation denominators:", np.round(t2_den, 3).tolist())  # -> [17.464, 8.349, 5.113]
t2_corr = t2_num / t2_den
print("correlations:", np.round(t2_corr, 3).tolist())  # -> [0.999, 0.958, -0.284]
t2_order = np.argsort(np.abs(t2_corr))[::-1]
print("filter order:", t2_order.tolist())  # -> [0, 1, 2]

plt.figure(figsize=(4.6, 3))
plt.bar(["x0", "x1", "x2"], np.abs(t2_corr), color="seagreen")
plt.ylabel("|correlation|")
plt.title("Toy 2 · filter score ranking")
plt.show()
assert int(t2_order[0]) == 0

▶ What you'll see: feature `x0` gets the tallest absolute-correlation bar and is ranked first by the filter.

### ✍️ Toy 3 · A wrapper searches subsets with validation loss

Wrapper selection fits a model for each candidate subset and scores that fitted model on validation data. The penalty lets a one-feature subset beat larger subsets with identical validation loss.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_Xtr = np.array([[0., 0., 1.], [1., 1., 0.], [2., 0., 1.], [3., 1., 0.], [4., 0., 1.], [5., 1., 0.]])
print("training shape:", t3_Xtr.shape)  # -> (6, 3)
t3_ytr = np.array([0., 1., 2., 3., 4., 5.])
print("training target:", t3_ytr.tolist())  # -> [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
t3_Xval = np.array([[1.5, 1., 0.], [2.5, 0., 1.], [4.5, 1., 0.]])
print("validation shape:", t3_Xval.shape)  # -> (3, 3)
t3_yval = np.array([1.5, 2.5, 4.5])
print("validation target:", t3_yval.tolist())  # -> [1.5, 2.5, 4.5]
t3_subsets = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2), (0, 1, 2)]
print("candidate subsets:", t3_subsets)  # -> [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2), (0, 1, 2)]
t3_scores = []
for t3_subset in t3_subsets:
    t3_A = np.column_stack((np.ones(len(t3_Xtr)), t3_Xtr[:, t3_subset]))
    t3_beta = np.linalg.lstsq(t3_A, t3_ytr, rcond=None)[0]
    t3_Aval = np.column_stack((np.ones(len(t3_Xval)), t3_Xval[:, t3_subset]))
    t3_pred = t3_Aval @ t3_beta
    t3_mse = float(np.mean((t3_yval - t3_pred) ** 2))
    t3_score = t3_mse + 0.03 * len(t3_subset)
    t3_scores.append(t3_score)
    print("subset", t3_subset, "mse", round(t3_mse, 3), "score", round(t3_score, 3))
t3_scores = np.array(t3_scores)
print("subset scores:", np.round(t3_scores, 3).tolist())  # -> [0.03, 1.613, 1.613, 0.06, 0.06, 1.643, 0.09]
t3_best_index = int(np.argmin(t3_scores))
print("best subset:", t3_subsets[t3_best_index])  # -> (0,)

plt.figure(figsize=(6, 3))
plt.bar([str(s) for s in t3_subsets], t3_scores, color="slateblue")
plt.xticks(rotation=35)
plt.ylabel("validation MSE + cost")
plt.title("Toy 3 · wrapper subset scores")
plt.show()
assert t3_subsets[t3_best_index] == (0,)

▶ What you'll see: subset `(0,)` wins because it fits validation perfectly while paying the smallest feature cost.

### ✍️ Toy 4 · Soft-thresholding makes embedded zeros

Embedded selection changes coefficients during fitting. A soft threshold subtracts a fixed amount from each magnitude and sets tiny coefficients exactly to zero.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_X = np.array([[0., 0., 0.], [1., 0., 1.], [0., 1., 1.], [1., 1., 0.], [2., 0., 1.], [0., 2., 0.]])
print("feature matrix shape:", t4_X.shape)  # -> (6, 3)
t4_A = np.column_stack((np.ones(len(t4_X)), t4_X))
print("design shape:", t4_A.shape)  # -> (6, 4)
t4_beta_true = np.array([1.0, 0.8, 0.25, 0.05])
print("true coefficients:", t4_beta_true.tolist())  # -> [1.0, 0.8, 0.25, 0.05]
t4_y = t4_A @ t4_beta_true
print("target:", np.round(t4_y, 3).tolist())  # -> [1.0, 1.85, 1.3, 2.05, 2.65, 1.5]
t4_beta_ls = np.linalg.lstsq(t4_A, t4_y, rcond=None)[0]
print("least-squares coefficients:", np.round(t4_beta_ls, 3).tolist())  # -> [1.0, 0.8, 0.25, 0.05]
t4_feature_coef = t4_beta_ls[1:]
print("feature coefficients:", np.round(t4_feature_coef, 3).tolist())  # -> [0.8, 0.25, 0.05]
t4_threshold = 0.10
print("soft threshold:", t4_threshold)  # -> 0.1
t4_shrunk = np.sign(t4_feature_coef) * np.maximum(np.abs(t4_feature_coef) - t4_threshold, 0.0)
print("shrunk coefficients:", np.round(t4_shrunk, 3).tolist())  # -> [0.7, 0.15, 0.0]
t4_selected = np.where(np.abs(t4_shrunk) > 0)[0]
print("selected features:", t4_selected.tolist())  # -> [0, 1]

plt.figure(figsize=(5, 3))
t4_xpos = np.arange(3)
plt.bar(t4_xpos - 0.18, t4_feature_coef, width=0.36, label="before", color="gray")
plt.bar(t4_xpos + 0.18, t4_shrunk, width=0.36, label="after", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(t4_xpos, ["x0", "x1", "x2"])
plt.title("Toy 4 · shrinkage selects nonzeros")
plt.legend()
plt.show()
assert np.array_equal(t4_selected, np.array([0, 1]))

▶ What you'll see: the smallest coefficient is shrunk to exactly zero, so only features `0` and `1` survive.

### ✍️ Toy 5 · Gap and stability change the final decision

A small absolute gap is easier to distrust than a large one. A stabilized version can become the safer choice once its full score is compared with the baseline and flexible alternatives.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_baseline = 0.31
print("baseline score:", t5_baseline)  # -> 0.31
t5_flexible = 0.34
print("flexible score:", t5_flexible)  # -> 0.34
t5_gap = t5_flexible - t5_baseline
print("absolute gap:", round(t5_gap, 3))  # -> 0.03
t5_relative_gap = t5_gap / t5_flexible
print("relative gap:", round(t5_relative_gap, 3))  # -> 0.088
t5_stable = 0.80 * t5_baseline
print("stabilized score:", round(t5_stable, 3))  # -> 0.248
t5_scores = np.array([t5_baseline, t5_flexible, t5_stable])
print("all scores:", np.round(t5_scores, 3).tolist())  # -> [0.31, 0.34, 0.248]
t5_labels = np.array(["baseline", "flexible", "stabilized"])
t5_winner = t5_labels[int(np.argmin(t5_scores))]
print("winner:", str(t5_winner))  # -> stabilized

plt.figure(figsize=(4.8, 3))
plt.bar(t5_labels, t5_scores, color=["gray", "orange", "seagreen"])
plt.ylabel("decision score")
plt.title("Toy 5 · stability wins the comparison")
plt.show()
assert t5_winner == "stabilized" and round(t5_relative_gap, 3) == 0.088

▶ What you'll see: the stabilized score is the lowest bar, and the baseline-flexible gap is only about `8.8%` of the flexible score.

## 🛠️ Setup

In [ ]:
import numpy as np  # Load NumPy for arrays, masks, least-squares fits, and reproducible simulations.
import matplotlib.pyplot as plt  # Load Matplotlib for the compact bars, lines, and heatmaps used to inspect feature selection.
np.random.seed(0)  # Make every stochastic example repeatable across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Average per-example losses

**Goal.** Turn individual validation losses into empirical risk, because every subset score starts by averaging errors over examples. We build it in 2 steps.

In [ ]:
losses_b1 = np.array([0.191, 0.070, 0.471])  # Store the three verified losses from the lesson source.
print("losses_b1:", losses_b1)  # Inspect the raw per-example errors before averaging.

▶ What you'll see: three small validation losses that will become one risk number.

In [ ]:
risk_b1 = float(np.mean(losses_b1))  # Average losses to compute empirical validation risk.
print("R_S:", round(risk_b1, 3))  # Inspect the subset's raw validation term.
assert round(risk_b1, 3) == 0.244  # Verify (0.191 + 0.070 + 0.471) / 3.
plt.figure(figsize=(4, 3))  # Create a compact loss chart.
plt.bar(["ex1", "ex2", "ex3", "mean"], list(losses_b1) + [risk_b1], color=["gray", "gray", "gray", "teal"])  # Compare individual losses with their average.
plt.title("Basic 1: empirical risk is an average")  # Title the plot.
plt.ylabel("loss")  # Label the loss scale.
plt.show()  # Display the plot.

▶ What you'll see: the mean bar sits between the three individual loss bars.

👀 Takeaway: empirical risk is the average loss, not the best or worst single example.

### Basic 2 — Add the feature cost

**Goal.** Compute the full selection score, because feature selection should penalize columns that add complexity or operating cost. We build it in 2 steps.

In [ ]:
risk_b2 = 0.244  # Use the verified raw validation risk.
cost_b2 = 0.090  # Use the lesson's feature-complexity cost.
print("risk:", risk_b2, "cost:", cost_b2)  # Inspect both score components.

▶ What you'll see: the raw fit term and the cost term are separate quantities.

In [ ]:
score_b2 = risk_b2 + cost_b2  # Add cost to risk to obtain the decision score.
print("selection score:", round(score_b2, 3))  # Inspect the final score used for ranking subsets.
assert round(score_b2, 3) == 0.334  # Verify the source arithmetic.
plt.figure(figsize=(4, 3))  # Create a score-decomposition chart.
plt.bar(["risk", "cost", "score"], [risk_b2, cost_b2, score_b2], color=["teal", "orange", "purple"])  # Plot components and total.
plt.title("Basic 2: risk + cost")  # Title the figure.
plt.ylabel("value")  # Label numeric scale.
plt.show()  # Display the chart.

▶ What you'll see: the full score is larger than raw risk because features are charged.

👀 Takeaway: dropping the cost term changes the selection rule.

### Basic 3 — Count selected features

**Goal.** Measure $|S|$, because the penalty grows with the number of chosen columns. We build it in 2 steps.

In [ ]:
selected_b3 = np.array([True, False, True, False, False])  # Mark a subset S inside five candidate features.
feature_ids_b3 = np.arange(len(selected_b3))  # Create feature indices for readable output.
print("selected feature ids:", feature_ids_b3[selected_b3])  # Inspect which columns are included.

▶ What you'll see: features 0 and 2 are selected.

In [ ]:
size_b3 = int(np.sum(selected_b3))  # Count True entries to compute |S|.
penalty_b3 = 0.04 * size_b3  # Charge lambda times the subset size.
print("|S|:", size_b3, "penalty:", round(penalty_b3, 3))  # Inspect the subset-size cost.
assert size_b3 == 2 and round(penalty_b3, 3) == 0.080  # Verify the count and penalty.
plt.figure(figsize=(4, 3))  # Create a binary subset plot.
plt.bar([f"x{i}" for i in feature_ids_b3], selected_b3.astype(int), color="seagreen")  # Show selected features as 1 and excluded as 0.
plt.title("Basic 3: subset indicator")  # Title the plot.
plt.ylim(0, 1.2)  # Keep the binary scale readable.
plt.show()  # Display the plot.

▶ What you'll see: selected columns have height 1, excluded columns have height 0.

👀 Takeaway: $|S|$ is just a count, but it is what turns simplicity into arithmetic.

### Basic 4 — Compute a feature-target correlation

**Goal.** Score one feature with Pearson correlation, because filter methods often rank columns by normalized co-movement with the target. We build it in 3 steps.

In [ ]:
x_b4 = np.array([0., 1., 2., 3., 4.])  # Define one candidate feature.
y_b4 = np.array([0.2, 1.2, 1.9, 3.1, 4.1])  # Define a target that mostly rises with the feature.
print("x:", x_b4)  # Inspect the feature values.
print("y:", y_b4)  # Inspect the target values.

▶ What you'll see: both arrays increase together.

In [ ]:
xc_b4 = x_b4 - np.mean(x_b4)  # Center the feature for correlation.
yc_b4 = y_b4 - np.mean(y_b4)  # Center the target for correlation.
print("centered x:", np.round(xc_b4, 2))  # Inspect deviations from feature mean.
print("centered y:", np.round(yc_b4, 2))  # Inspect deviations from target mean.

In [ ]:
corr_b4 = float(np.dot(xc_b4, yc_b4) / (np.linalg.norm(xc_b4) * np.linalg.norm(yc_b4)))  # Compute Pearson correlation from dot product and norms.
print("correlation:", round(corr_b4, 3))  # Inspect the filter score.
assert corr_b4 > 0.99  # Verify this feature is almost perfectly aligned with y.
plt.figure(figsize=(4, 3))  # Create a scatter plot.
plt.scatter(x_b4, y_b4, color="teal", s=70)  # Plot feature against target.
plt.title("Basic 4: strong filter signal")  # Title the plot.
plt.xlabel("feature x")  # Label feature axis.
plt.ylabel("target y")  # Label target axis.
plt.show()  # Display the scatter.

▶ What you'll see: points lie close to an increasing line, matching the near-1 correlation.

👀 Takeaway: correlation is a scale-normalized filter score for one feature at a time.

### Basic 5 — Rank features by filter score

**Goal.** Choose the top filter features, because a filter method screens columns before model fitting. We build it in 2 steps.

In [ ]:
scores_b5 = np.array([0.98, 0.42, 0.07, 0.63])  # Store absolute feature-target scores.
print("filter scores:", scores_b5)  # Inspect all feature scores before ranking.

▶ What you'll see: feature 0 is strongest, feature 2 is weakest.

In [ ]:
order_b5 = np.argsort(scores_b5)[::-1]  # Sort feature indices from largest score to smallest.
top2_b5 = order_b5[:2]  # Keep the top two filter-ranked features.
print("ranked features:", order_b5)  # Inspect full ranking.
print("top two:", top2_b5)  # Inspect selected feature ids.
assert np.array_equal(top2_b5, np.array([0, 3]))  # Verify the top-2 rule.
plt.figure(figsize=(4, 3))  # Create a filter-ranking plot.
plt.bar(["x0", "x1", "x2", "x3"], scores_b5, color="purple")  # Draw score bars.
plt.title("Basic 5: filter ranking")  # Title the chart.
plt.ylabel("absolute score")  # Label score axis.
plt.show()  # Display the chart.

▶ What you'll see: the selected features are exactly the two tallest bars.

👀 Takeaway: filters are fast because selection uses feature scores, not repeated model fits.

### Basic 6 — Fit a tiny linear model on selected columns

**Goal.** Train on a chosen subset, because wrapper and embedded methods must evaluate models using only selected columns. We build it in 3 steps.

In [ ]:
X_b6 = np.array([[0., 0.], [1., 0.], [2., 1.], [3., 1.]])  # Define two candidate features.
y_b6 = np.array([0.1, 1.0, 2.2, 3.0])  # Define a small regression target.
S_b6 = np.array([0])  # Select only the first feature.
print("selected columns:", S_b6)  # Inspect the subset.

▶ What you'll see: only feature 0 is allowed into the model.

In [ ]:
A_b6 = np.c_[np.ones(len(X_b6)), X_b6[:, S_b6]]  # Add an intercept column to the selected feature matrix.
beta_b6 = np.linalg.lstsq(A_b6, y_b6, rcond=None)[0]  # Fit least squares from scratch.
print("beta:", np.round(beta_b6, 3))  # Inspect intercept and slope.

In [ ]:
pred_b6 = A_b6 @ beta_b6  # Predict training targets using selected columns only.
mse_b6 = float(np.mean((y_b6 - pred_b6) ** 2))  # Compute mean squared training error.
print("MSE:", round(mse_b6, 4))  # Inspect fit quality.
assert mse_b6 < 0.02  # Verify the selected feature explains this toy target well.
plt.figure(figsize=(4, 3))  # Create a fit diagnostic plot.
plt.plot(y_b6, marker="o", label="actual")  # Plot true values.
plt.plot(pred_b6, marker="s", label="predicted")  # Plot fitted values.
plt.title("Basic 6: fit on selected columns")  # Title the chart.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: predictions closely track the target even with one selected feature.

👀 Takeaway: once $S$ is chosen, the model can only use columns inside $S$.

### Basic 7 — Score a wrapper candidate

**Goal.** Combine validation MSE and subset cost for one candidate, because wrapper selection ranks subsets by full decision score. We build it in 2 steps.

In [ ]:
val_errors_b7 = np.array([0.10, -0.20, 0.05])  # Store prediction residuals on validation examples.
S_size_b7 = 2  # Candidate subset uses two features.
lam_b7 = 0.03  # Cost per selected feature.
print("validation residuals:", val_errors_b7)  # Inspect errors before squaring.

▶ What you'll see: three signed residuals that become nonnegative squared losses.

In [ ]:
mse_b7 = float(np.mean(val_errors_b7 ** 2))  # Average squared validation errors.
score_b7 = mse_b7 + lam_b7 * S_size_b7  # Add subset-size cost.
print("validation MSE:", round(mse_b7, 4), "wrapper score:", round(score_b7, 4))  # Inspect full score.
assert round(mse_b7, 4) == 0.0175 and round(score_b7, 4) == 0.0775  # Verify arithmetic.
plt.figure(figsize=(4, 3))  # Create a score component chart.
plt.bar(["MSE", "λ|S|", "score"], [mse_b7, lam_b7 * S_size_b7, score_b7], color=["teal", "orange", "purple"])  # Show score pieces.
plt.title("Basic 7: wrapper score")  # Title the plot.
plt.show()  # Display the plot.

▶ What you'll see: feature cost dominates this toy candidate's small validation MSE.

👀 Takeaway: wrapper selection can reject a better fit if the extra features are not worth their cost.

### Basic 8 — Soft-threshold a coefficient

**Goal.** Shrink one coefficient toward zero, because embedded selection can remove weak features during fitting. We build it in 2 steps.

In [ ]:
coef_b8 = np.array([0.85, 0.18, -0.04])  # Store three fitted feature coefficients.
threshold_b8 = 0.20  # Define the shrinkage threshold.
print("raw coefficients:", coef_b8)  # Inspect coefficients before shrinkage.

▶ What you'll see: one strong positive coefficient and two small coefficients.

In [ ]:
shrunk_b8 = np.sign(coef_b8) * np.maximum(np.abs(coef_b8) - threshold_b8, 0.0)  # Apply soft thresholding.
selected_b8 = np.abs(shrunk_b8) > 0  # Keep nonzero coefficients.
print("shrunk coefficients:", np.round(shrunk_b8, 3))  # Inspect post-penalty values.
print("selected mask:", selected_b8.astype(int))  # Inspect embedded selection result.
assert np.array_equal(selected_b8, np.array([True, False, False]))  # Verify only the strong coefficient survives.
plt.figure(figsize=(4, 3))  # Create a shrinkage comparison plot.
plt.bar(["c0", "c1", "c2"], shrunk_b8, color="seagreen")  # Plot shrunk coefficients.
plt.axhline(0, color="black", linewidth=0.8)  # Add zero reference.
plt.title("Basic 8: soft-threshold selection")  # Title the plot.
plt.show()  # Display the chart.

▶ What you'll see: weak coefficients become exactly zero.

👀 Takeaway: embedded methods can express exclusion as a coefficient equal to zero.

### Basic 9 — Compare two decision scores

**Goal.** Compute absolute and relative gaps, because a tiny score difference may not justify changing the selected subset. We build it in 2 steps.

In [ ]:
score_a_b9 = 0.334  # Baseline subset decision score.
score_b_b9 = 0.374  # More flexible alternative decision score.
print("scores:", score_a_b9, score_b_b9)  # Inspect both candidates.

▶ What you'll see: the baseline candidate has the lower score.

In [ ]:
gap_b9 = score_b_b9 - score_a_b9  # Absolute advantage of the lower-scoring candidate.
relative_b9 = gap_b9 / score_b_b9  # Scale the gap by the alternative's score.
print("gap:", round(gap_b9, 3), "relative:", round(relative_b9, 3))  # Inspect evidence size.
assert round(gap_b9, 3) == 0.040 and round(relative_b9, 3) == 0.107  # Verify lesson numbers.
plt.figure(figsize=(4, 3))  # Create a comparison bar chart.
plt.bar(["baseline", "flexible"], [score_a_b9, score_b_b9], color=["teal", "orange"])  # Plot scores.
plt.title("Basic 9: score gap")  # Title the chart.
plt.ylabel("lower is better")  # Label decision direction.
plt.show()  # Display the chart.

▶ What you'll see: the gap is visible but modest.

👀 Takeaway: validation gaps are evidence, not decoration.

### Basic 10 — Pick the minimum score

**Goal.** Choose the lowest full score among candidates, because feature selection is an optimization over subsets and settings. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.334, 0.374, 0.267])  # Store baseline, flexible, and stabilized scores.
labels_b10 = np.array(["baseline", "flexible", "stabilized"])  # Name the candidates.
print("candidate scores:", dict(zip(labels_b10, np.round(scores_b10, 3))))  # Inspect the decision table.

▶ What you'll see: three possible scores on the same scale.

In [ ]:
winner_idx_b10 = int(np.argmin(scores_b10))  # Find the lowest score.
winner_b10 = labels_b10[winner_idx_b10]  # Read the winning label.
print("winner:", winner_b10, "score:", scores_b10[winner_idx_b10])  # Inspect the selected candidate.
assert winner_b10 == "stabilized" and round(float(scores_b10[winner_idx_b10]), 3) == 0.267  # Verify the final minimum.
plt.figure(figsize=(4, 3))  # Create final-decision plot.
plt.bar(labels_b10, scores_b10, color=["gray", "orange", "seagreen"])  # Compare candidates.
plt.title("Basic 10: minimum score wins")  # Title the chart.
plt.ylabel("decision score")  # Label score scale.
plt.show()  # Display the chart.

▶ What you'll see: the stabilized option has the shortest bar.

👀 Takeaway: the winner is the lowest complete decision score, not the flashiest raw fit.

## 🟡 Easy

### Easy 1 — Implement a complete filter selector

**Goal.** Rank four features by absolute correlation and keep the top two, because filter selection is a fast first pass before model fitting. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0., 0., 1., 2.], [1., 0., 1., 1.], [2., 1., 0., 0.], [3., 1., 0., 1.], [4., 2., 1., 2.], [5., 2., 0., 1.]])  # Candidate features.
y_e1 = np.array([0.2, 1.0, 2.1, 2.9, 4.2, 5.1])  # Target mostly follows features 0 and 1.
print("X shape:", X_e1.shape)  # Inspect the feature table dimensions.

▶ What you'll see: six examples and four candidate columns.

In [ ]:
Xc_e1 = X_e1 - X_e1.mean(axis=0)  # Center features.
yc_e1 = y_e1 - y_e1.mean()  # Center target.
corr_e1 = (Xc_e1.T @ yc_e1) / np.sqrt(np.sum(Xc_e1 ** 2, axis=0) * np.sum(yc_e1 ** 2))  # Compute correlations from scratch.
rank_e1 = np.argsort(np.abs(corr_e1))[::-1]  # Sort by absolute correlation.
print("correlations:", np.round(corr_e1, 3))  # Inspect filter scores.
print("rank:", rank_e1)  # Inspect feature order.

In [ ]:
top2_e1 = rank_e1[:2]  # Select two strongest filter features.
print("selected features:", top2_e1)  # Inspect top-2 selection.
assert np.array_equal(top2_e1, np.array([0, 1]))  # Verify expected filter choice.
plt.figure(figsize=(5, 3))  # Create a filter-score chart.
plt.bar([f"x{i}" for i in range(X_e1.shape[1])], np.abs(corr_e1), color="teal")  # Plot absolute correlations.
plt.title("Easy 1: filter selector")  # Title the chart.
plt.ylabel("|corr(x_j, y)|")  # Label score axis.
plt.show()  # Display the chart.

▶ What you'll see: features 0 and 1 have the tallest correlation bars.

👀 Takeaway: filter methods are simple rankings based on feature-target statistics.

### Easy 2 — Exhaustively search wrapper subsets

**Goal.** Try every nonempty subset of three features, because wrapper selection evaluates the actual model and validation score for each subset. We build it in 4 steps.

In [ ]:
Xtr_e2 = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.]])  # Training features.
ytr_e2 = np.array([0.1, 1.1, 2.0, 2.9, 4.0, 5.2])  # Training target.
Xval_e2 = np.array([[1.5, 0.5, 0.0], [3.5, 1.5, 1.0], [4.5, 2.0, 0.0]])  # Validation features.
yval_e2 = np.array([1.55, 3.55, 4.75])  # Validation target.
print("train/val rows:", len(Xtr_e2), len(Xval_e2))  # Inspect split sizes.

▶ What you'll see: a tiny supervised split for wrapper scoring.

In [ ]:
subsets_e2 = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2), (0, 1, 2)]  # Enumerate all nonempty subsets.
lam_e2 = 0.03  # Cost per selected feature.
scores_e2 = []  # Store decision scores.
print("subsets:", subsets_e2)  # Inspect candidates.

In [ ]:
for S_e2 in subsets_e2:  # Evaluate each candidate subset.
    A_e2 = np.c_[np.ones(len(Xtr_e2)), Xtr_e2[:, S_e2]]  # Select training columns and add intercept.
    beta_e2 = np.linalg.lstsq(A_e2, ytr_e2, rcond=None)[0]  # Fit the subset model.
    Aval_e2 = np.c_[np.ones(len(Xval_e2)), Xval_e2[:, S_e2]]  # Select validation columns.
    mse_e2 = float(np.mean((yval_e2 - Aval_e2 @ beta_e2) ** 2))  # Compute validation risk.
    scores_e2.append(mse_e2 + lam_e2 * len(S_e2))  # Add subset-size penalty.
print("scores:", np.round(scores_e2, 3))  # Inspect all wrapper scores.

In [ ]:
best_e2 = int(np.argmin(scores_e2))  # Locate the lowest wrapper score.
print("best subset:", subsets_e2[best_e2], "score:", round(scores_e2[best_e2], 3))  # Inspect the selected subset.
assert subsets_e2[best_e2] == (0,)  # Verify the wrapper result on this toy split.
plt.figure(figsize=(6, 3))  # Create a subset-score plot.
plt.bar([str(s) for s in subsets_e2], scores_e2, color="purple")  # Plot wrapper scores.
plt.xticks(rotation=35)  # Rotate labels for readability.
plt.title("Easy 2: exhaustive wrapper search")  # Title the plot.
plt.ylabel("MSE + λ|S|")  # Label score axis.
plt.show()  # Display the chart.

▶ What you'll see: the best subset is the lowest bar; here the cost favors one strong feature over a larger set.

👀 Takeaway: wrappers spend compute to measure how subsets behave inside the actual model.

### Easy 3 — Compare raw fit with penalized fit

**Goal.** Show how a lower raw validation MSE can lose after feature cost, because selection should use the full score. We build it in 3 steps.

In [ ]:
names_e3 = np.array(["small", "large"])  # Compare a small and a larger subset.
val_mse_e3 = np.array([0.060, 0.035])  # Larger subset has better raw validation fit.
size_e3 = np.array([1, 3])  # Larger subset uses more features.
lam_e3 = 0.020  # Feature cost.
print("raw MSE:", val_mse_e3, "sizes:", size_e3)  # Inspect ingredients.

▶ What you'll see: the large subset looks better if you only read raw MSE.

In [ ]:
penalty_e3 = lam_e3 * size_e3  # Compute λ|S| for both candidates.
score_e3 = val_mse_e3 + penalty_e3  # Add cost to raw validation fit.
print("penalties:", penalty_e3)  # Inspect cost terms.
print("full scores:", score_e3)  # Inspect decision scores.
assert np.argmin(score_e3) == 0  # The small subset wins after cost.

In [ ]:
x_e3 = np.arange(len(names_e3))  # Bar positions.
plt.figure(figsize=(5, 3))  # Create grouped comparison plot.
plt.bar(x_e3 - 0.18, val_mse_e3, width=0.36, label="raw MSE", color="gray")  # Plot raw fit.
plt.bar(x_e3 + 0.18, score_e3, width=0.36, label="MSE + cost", color="teal")  # Plot penalized score.
plt.xticks(x_e3, names_e3)  # Label candidates.
plt.title("Easy 3: raw fit can mislead")  # Title the chart.
plt.legend()  # Show legend.
plt.show()  # Display the plot.

▶ What you'll see: the large subset has lower raw MSE but higher penalized score.

👀 Takeaway: feature selection ranks candidates on the same full decision scale.

### Easy 4 — Embedded selection with shrinkage

**Goal.** Fit all features, shrink coefficients, and read selected nonzeros, because embedded selection couples fitting and feature choice. We build it in 4 steps.

In [ ]:
X_e4 = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.]])  # Candidate features.
y_e4 = np.array([0.1, 1.1, 2.0, 2.9, 4.0, 5.2])  # Target.
A_e4 = np.c_[np.ones(len(X_e4)), X_e4]  # Add intercept before least squares.
print("design shape:", A_e4.shape)  # Inspect design matrix size.

▶ What you'll see: the model has one intercept plus three feature columns.

In [ ]:
beta_e4 = np.linalg.lstsq(A_e4, y_e4, rcond=None)[0]  # Fit least-squares coefficients.
coefs_e4 = beta_e4[1:]  # Drop intercept for feature selection.
print("raw coefficients:", np.round(coefs_e4, 3))  # Inspect feature weights.

In [ ]:
threshold_e4 = 0.07  # Shrinkage threshold.
shrunk_e4 = np.sign(coefs_e4) * np.maximum(np.abs(coefs_e4) - threshold_e4, 0.0)  # Soft-threshold coefficients.
selected_e4 = np.where(np.abs(shrunk_e4) > 0)[0]  # Nonzero coefficients are selected.
print("shrunk:", np.round(shrunk_e4, 3))  # Inspect shrunk weights.
print("selected:", selected_e4)  # Inspect embedded feature choice.
assert np.array_equal(selected_e4, np.array([0, 1]))  # Verify expected nonzero features.

In [ ]:
plt.figure(figsize=(5, 3))  # Create coefficient plot.
plt.bar(["x0", "x1", "x2"], shrunk_e4, color="seagreen")  # Plot shrunk coefficients.
plt.axhline(0, color="black", linewidth=0.8)  # Show zero line.
plt.title("Easy 4: embedded nonzeros")  # Title the chart.
plt.show()  # Display the plot.

▶ What you'll see: nonzero bars identify selected features after shrinkage.

👀 Takeaway: embedded methods can make selection an outcome of the fitted objective.

### Easy 5 — Validate stability with resampled filter rankings

**Goal.** Recompute filter rankings on bootstrap samples, because a useful selected feature should not appear only because of one lucky split. We build it in 4 steps.

In [ ]:
X_e5 = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.], [6., 3., 1.], [7., 3., 0.]])  # Features with two stable signals.
y_e5 = np.array([0.2, 1.0, 2.1, 2.9, 4.2, 5.1, 6.0, 7.1])  # Target.
rng_e5 = np.random.default_rng(5)  # Local generator for reproducible resamples.
print("rows:", len(X_e5))  # Inspect sample size.

▶ What you'll see: a small dataset ready for repeated resampling.

In [ ]:
counts_e5 = np.zeros(X_e5.shape[1], dtype=int)  # Count how often each feature is selected.
for rep_e5 in range(40):  # Run many bootstrap rankings.
    idx_e5 = rng_e5.integers(0, len(X_e5), size=len(X_e5))  # Sample rows with replacement.
    Xs_e5 = X_e5[idx_e5]  # Bootstrap features.
    ys_e5 = y_e5[idx_e5]  # Bootstrap target.
    Xc_e5 = Xs_e5 - Xs_e5.mean(axis=0)  # Center features.
    yc_e5 = ys_e5 - ys_e5.mean()  # Center target.
    den_e5 = np.sqrt(np.sum(Xc_e5 ** 2, axis=0) * np.sum(yc_e5 ** 2))  # Correlation denominators.
    corr_e5 = np.divide(Xc_e5.T @ yc_e5, den_e5, out=np.zeros(X_e5.shape[1]), where=den_e5 > 0)  # Guarded correlations.
    counts_e5[np.argmax(np.abs(corr_e5))] += 1  # Count the top feature.
print("top-feature counts:", counts_e5)  # Inspect selection frequencies.

In [ ]:
stability_e5 = counts_e5 / np.sum(counts_e5)  # Convert counts into frequencies.
print("selection frequencies:", np.round(stability_e5, 2))  # Inspect stability across resamples.
assert int(np.argmax(stability_e5)) == 0  # Feature 0 is most stable.

In [ ]:
plt.figure(figsize=(4, 3))  # Create stability bar chart.
plt.bar(["x0", "x1", "x2"], stability_e5, color="teal")  # Plot selection frequency.
plt.title("Easy 5: resampled filter stability")  # Title the chart.
plt.ylabel("top-selection frequency")  # Label frequency axis.
plt.ylim(0, 1)  # Bound the frequency scale.
plt.show()  # Display the chart.

▶ What you'll see: the most stable feature is selected in most bootstrap samples.

👀 Takeaway: stability checks whether a feature choice survives sampling variation.

## 🔴 Advanced

### Advanced 1 — Forward stepwise wrapper selection

**Goal.** Add features greedily only when they improve penalized validation score, because exhaustive wrapper search becomes expensive as feature count grows. We build it in 5 steps.

In [ ]:
Xtr_a1 = np.array([[0., 0., 1., 2.], [1., 0., 1., 1.], [2., 1., 0., 0.], [3., 1., 0., 1.], [4., 2., 1., 2.], [5., 2., 0., 1.], [6., 3., 1., 0.]])  # Training features.
ytr_a1 = np.array([0.1, 1.0, 2.1, 2.9, 4.1, 5.0, 6.2])  # Training target.
Xval_a1 = np.array([[1.5, 0.5, 0., 1.], [3.5, 1.5, 1., 2.], [5.5, 2.5, 0., 0.]])  # Validation features.
yval_a1 = np.array([1.55, 3.65, 5.65])  # Validation target.
print("candidate feature count:", Xtr_a1.shape[1])  # Inspect search dimension.

▶ What you'll see: four features are available for greedy selection.

In [ ]:
selected_a1 = []  # Start with no selected features.
remaining_a1 = list(range(Xtr_a1.shape[1]))  # All features are initially available.
lam_a1 = 0.025  # Cost per selected feature.
history_a1 = []  # Store chosen score after each step.
print("start selected:", selected_a1)  # Inspect empty subset.

In [ ]:
for step_a1 in range(3):  # Try adding up to three features.
    trial_scores_a1 = []  # Store candidate additions.
    for j_a1 in remaining_a1:  # Test each remaining feature.
        S_a1 = selected_a1 + [j_a1]  # Candidate subset after adding this feature.
        A_a1 = np.c_[np.ones(len(Xtr_a1)), Xtr_a1[:, S_a1]]  # Training design for candidate subset.
        beta_a1 = np.linalg.lstsq(A_a1, ytr_a1, rcond=None)[0]  # Fit candidate model.
        Aval_a1 = np.c_[np.ones(len(Xval_a1)), Xval_a1[:, S_a1]]  # Validation design.
        mse_a1 = float(np.mean((yval_a1 - Aval_a1 @ beta_a1) ** 2))  # Validation MSE.
        trial_scores_a1.append(mse_a1 + lam_a1 * len(S_a1))  # Penalized score.
    best_local_a1 = int(np.argmin(trial_scores_a1))  # Best addition this round.
    selected_a1.append(remaining_a1.pop(best_local_a1))  # Accept the best feature.
    history_a1.append(trial_scores_a1[best_local_a1])  # Record score.
print("selected path:", selected_a1)  # Inspect greedy feature order.
print("history:", np.round(history_a1, 3))  # Inspect score path.

In [ ]:
best_len_a1 = int(np.argmin(history_a1)) + 1  # Pick the path length with best penalized score.
final_subset_a1 = selected_a1[:best_len_a1]  # Truncate greedy path to best length.
print("final subset:", final_subset_a1)  # Inspect selected greedy subset.
assert 0 in final_subset_a1  # Verify the main signal feature is retained.

In [ ]:
plt.figure(figsize=(5, 3))  # Create greedy path plot.
plt.plot(np.arange(1, len(history_a1) + 1), history_a1, marker="o", color="purple")  # Plot score after each accepted feature.
plt.title("Advanced 1: forward wrapper path")  # Title the chart.
plt.xlabel("features selected")  # Label path length.
plt.ylabel("validation MSE + λ|S|")  # Label score axis.
plt.show()  # Display the plot.

▶ What you'll see: penalized score improves early, then may flatten when extra features cost more than they help.

👀 Takeaway: forward selection approximates wrapper search by adding the best next feature instead of enumerating every subset.

### Advanced 2 — Nested validation for subset choice and final test

**Goal.** Separate subset selection from final testing, because using the same data to choose and report performance gives an optimistic estimate. We build it in 5 steps.

In [ ]:
X_a2 = np.array([[0., 0., 1.], [1., 0., 1.], [2., 1., 0.], [3., 1., 0.], [4., 2., 1.], [5., 2., 0.], [6., 3., 1.], [7., 3., 0.], [8., 4., 1.]])  # Full toy design.
y_a2 = np.array([0.2, 1.1, 2.0, 2.9, 4.1, 5.1, 6.0, 7.2, 8.0])  # Full toy target.
train_a2 = np.arange(0, 5)  # Inner training indices.
val_a2 = np.arange(5, 7)  # Subset-selection validation indices.
test_a2 = np.arange(7, 9)  # Final untouched test indices.
print("split sizes:", len(train_a2), len(val_a2), len(test_a2))  # Inspect split sizes.

▶ What you'll see: train, validation, and test rows are distinct.

In [ ]:
subsets_a2 = [(0,), (1,), (2,), (0, 1), (0, 2), (1, 2), (0, 1, 2)]  # Candidate subsets.
val_scores_a2 = []  # Store inner validation scores.
for S_a2 in subsets_a2:  # Evaluate each subset on validation rows.
    Atrain_a2 = np.c_[np.ones(len(train_a2)), X_a2[train_a2][:, S_a2]]  # Train design.
    beta_a2 = np.linalg.lstsq(Atrain_a2, y_a2[train_a2], rcond=None)[0]  # Fit on training rows only.
    Aval_a2 = np.c_[np.ones(len(val_a2)), X_a2[val_a2][:, S_a2]]  # Validation design.
    mse_a2 = float(np.mean((y_a2[val_a2] - Aval_a2 @ beta_a2) ** 2))  # Validation MSE.
    val_scores_a2.append(mse_a2 + 0.03 * len(S_a2))  # Penalized validation score.
print("validation scores:", np.round(val_scores_a2, 3))  # Inspect subset-selection scores.

In [ ]:
best_a2 = int(np.argmin(val_scores_a2))  # Choose subset using validation only.
S_best_a2 = subsets_a2[best_a2]  # Read best subset.
print("chosen subset:", S_best_a2)  # Inspect subset choice.
assert 0 in S_best_a2  # Main signal should be included.

In [ ]:
fit_idx_a2 = np.r_[train_a2, val_a2]  # Refit on train + validation after choosing S.
Afit_a2 = np.c_[np.ones(len(fit_idx_a2)), X_a2[fit_idx_a2][:, S_best_a2]]  # Fit design for chosen subset.
beta_final_a2 = np.linalg.lstsq(Afit_a2, y_a2[fit_idx_a2], rcond=None)[0]  # Final fit before test.
Atest_a2 = np.c_[np.ones(len(test_a2)), X_a2[test_a2][:, S_best_a2]]  # Test design.
test_mse_a2 = float(np.mean((y_a2[test_a2] - Atest_a2 @ beta_final_a2) ** 2))  # Final test MSE.
print("final test MSE:", round(test_mse_a2, 3))  # Inspect unbiased-ish final score.

In [ ]:
plt.figure(figsize=(5, 3))  # Create validation-score plot.
plt.bar([str(s) for s in subsets_a2], val_scores_a2, color="teal")  # Show inner validation selection scores.
plt.xticks(rotation=35)  # Rotate labels.
plt.title("Advanced 2: choose on validation, report on test")  # Title chart.
plt.ylabel("inner score")  # Label score axis.
plt.show()  # Display the plot.

▶ What you'll see: the subset is chosen on validation scores, while the final test MSE is computed only once afterward.

👀 Takeaway: nested evaluation protects the final reported score from subset-search overfitting.

### Advanced 3 — Detect redundant correlated features

**Goal.** Penalize selecting duplicate signals, because two highly correlated features can add cost without adding much new information. We build it in 4 steps.

In [ ]:
x0_a3 = np.array([0., 1., 2., 3., 4., 5.])  # First signal feature.
x1_a3 = x0_a3 + np.array([0.0, 0.1, -0.1, 0.0, 0.1, -0.1])  # Near-duplicate signal feature.
x2_a3 = np.array([1., 0., 1., 0., 1., 0.])  # Different pattern.
X_a3 = np.c_[x0_a3, x1_a3, x2_a3]  # Combine features.
print("X_a3 shape:", X_a3.shape)  # Inspect feature table.

▶ What you'll see: features 0 and 1 are deliberately similar.

In [ ]:
Xc_a3 = X_a3 - X_a3.mean(axis=0)  # Center features.
corrmat_a3 = (Xc_a3.T @ Xc_a3) / np.sqrt(np.outer(np.sum(Xc_a3 ** 2, axis=0), np.sum(Xc_a3 ** 2, axis=0)))  # Feature-feature correlations.
print("feature correlation matrix:\n", np.round(corrmat_a3, 3))  # Inspect redundancy.
assert corrmat_a3[0, 1] > 0.99  # Verify x0 and x1 are near duplicates.

In [ ]:
candidate_a3 = np.array([0, 1])  # Tempting subset with two redundant signals.
redundancy_a3 = abs(corrmat_a3[candidate_a3[0], candidate_a3[1]])  # Pairwise redundancy score.
base_score_a3 = 0.080  # Suppose validation MSE is small.
redundant_score_a3 = base_score_a3 + 0.03 * len(candidate_a3) + 0.05 * redundancy_a3  # Add size and redundancy costs.
print("redundancy:", round(redundancy_a3, 3), "score:", round(redundant_score_a3, 3))  # Inspect adjusted score.

In [ ]:
plt.figure(figsize=(4, 3))  # Create heatmap figure.
plt.imshow(corrmat_a3, vmin=-1, vmax=1, cmap="coolwarm")  # Plot correlations from -1 to 1.
plt.colorbar(label="correlation")  # Add scale.
plt.xticks(range(3), ["x0", "x1", "x2"])  # Label x axis.
plt.yticks(range(3), ["x0", "x1", "x2"])  # Label y axis.
plt.title("Advanced 3: redundancy heatmap")  # Title plot.
plt.show()  # Display heatmap.

▶ What you'll see: the x0-x1 cell is almost perfectly correlated, warning that the pair may be redundant.

👀 Takeaway: feature cost can include redundancy, not only the raw number of selected columns.

### Advanced 4 — Permutation importance from validation loss

**Goal.** Estimate how much each selected feature matters by shuffling it on validation data, because useful features should hurt validation performance when their information is destroyed. We build it in 5 steps.

In [ ]:
Xtr_a4 = np.array([[0., 0.], [1., 0.], [2., 1.], [3., 1.], [4., 2.], [5., 2.]])  # Training features.
ytr_a4 = np.array([0.1, 1.0, 2.1, 2.9, 4.0, 5.2])  # Training target.
Xval_a4 = np.array([[1.5, 0.5], [3.5, 1.5], [4.5, 2.0]])  # Validation features.
yval_a4 = np.array([1.55, 3.55, 4.75])  # Validation target.
print("selected feature set: [0, 1]")  # Inspect chosen subset.

▶ What you'll see: two selected features will be stress-tested.

In [ ]:
A_a4 = np.c_[np.ones(len(Xtr_a4)), Xtr_a4]  # Add intercept.
beta_a4 = np.linalg.lstsq(A_a4, ytr_a4, rcond=None)[0]  # Fit model on selected features.
base_pred_a4 = np.c_[np.ones(len(Xval_a4)), Xval_a4] @ beta_a4  # Predict validation rows.
base_mse_a4 = float(np.mean((yval_a4 - base_pred_a4) ** 2))  # Baseline validation MSE.
print("base MSE:", round(base_mse_a4, 4))  # Inspect unshuffled score.

In [ ]:
rng_a4 = np.random.default_rng(4)  # Local generator for reproducible permutations.
importances_a4 = []  # Store MSE increases.
for j_a4 in range(Xval_a4.shape[1]):  # Shuffle each selected feature separately.
    Xperm_a4 = Xval_a4.copy()  # Copy validation features.
    Xperm_a4[:, j_a4] = rng_a4.permutation(Xperm_a4[:, j_a4])  # Destroy the column's alignment with y.
    pred_perm_a4 = np.c_[np.ones(len(Xperm_a4)), Xperm_a4] @ beta_a4  # Predict with permuted feature.
    mse_perm_a4 = float(np.mean((yval_a4 - pred_perm_a4) ** 2))  # Compute permuted MSE.
    importances_a4.append(mse_perm_a4 - base_mse_a4)  # Importance is the loss increase.
print("permutation importances:", np.round(importances_a4, 4))  # Inspect loss increases.

In [ ]:
important_a4 = np.array(importances_a4) > 0.01  # Flag features whose shuffle hurts meaningfully.
print("important mask:", important_a4.astype(int))  # Inspect feature importance flags.
assert np.any(important_a4)  # At least one selected feature should matter.

In [ ]:
plt.figure(figsize=(4, 3))  # Create importance plot.
plt.bar(["x0", "x1"], importances_a4, color="orange")  # Plot MSE increase by feature.
plt.axhline(0, color="black", linewidth=0.8)  # Show zero line.
plt.title("Advanced 4: permutation importance")  # Title chart.
plt.ylabel("MSE increase after shuffle")  # Label importance axis.
plt.show()  # Display plot.

▶ What you'll see: shuffling the more useful feature creates a larger validation-loss increase.

👀 Takeaway: permutation importance checks selected features by asking whether validation loss depends on them.

### Advanced 5 — Tune λ for the selection objective

**Goal.** Sweep the feature-cost strength $\lambda$, because the best subset size depends on how much we value simplicity versus validation fit. We build it in 5 steps.

In [ ]:
subset_names_a5 = np.array(["one", "two", "three", "four"])  # Candidate subset sizes.
val_mse_a5 = np.array([0.120, 0.070, 0.055, 0.052])  # Raw validation MSE improves with more features.
sizes_a5 = np.array([1, 2, 3, 4])  # Number of selected features in each candidate.
lambdas_a5 = np.array([0.000, 0.010, 0.030, 0.060])  # Cost strengths to test.
print("raw validation MSE:", val_mse_a5)  # Inspect fit-only curve.

▶ What you'll see: larger subsets have slightly lower raw validation MSE.

In [ ]:
score_grid_a5 = []  # Store one score row per lambda.
for lam_a5 in lambdas_a5:  # Sweep feature-cost strengths.
    score_grid_a5.append(val_mse_a5 + lam_a5 * sizes_a5)  # Compute R_val + λ|S| for each subset.
score_grid_a5 = np.array(score_grid_a5)  # Convert to matrix for indexing and plotting.
print("score grid:\n", np.round(score_grid_a5, 3))  # Inspect all decisions.

In [ ]:
best_idx_a5 = np.argmin(score_grid_a5, axis=1)  # Best subset for each lambda.
best_names_a5 = subset_names_a5[best_idx_a5]  # Convert best indices to labels.
print("best subset by lambda:", list(zip(lambdas_a5, best_names_a5)))  # Inspect how choices change.
assert best_names_a5[0] == "four" and best_names_a5[-1] == "one"  # Verify low λ favors larger subsets, high λ favors smaller.

In [ ]:
stable_score_a5 = 0.80 * 0.334  # Recompute the source lesson's stabilizing example.
print("source stabilized score:", round(stable_score_a5, 3))  # Inspect stabilized score.
assert round(stable_score_a5, 3) == 0.267  # Verify source arithmetic.

In [ ]:
plt.figure(figsize=(5, 3))  # Create lambda-sweep figure.
for k_a5, name_a5 in enumerate(subset_names_a5):  # Plot each subset's score as lambda changes.
    plt.plot(lambdas_a5, score_grid_a5[:, k_a5], marker="o", label=name_a5)  # Draw score curve.
plt.title("Advanced 5: λ changes selected subset size")  # Title chart.
plt.xlabel("λ")  # Label cost strength.
plt.ylabel("R_val + λ|S|")  # Label selection score.
plt.legend()  # Show subset labels.
plt.show()  # Display plot.

▶ What you'll see: as λ increases, the winning subset shifts from larger to smaller.

👀 Takeaway: λ is the explicit knob that decides how much validation improvement an added feature must buy.